In [3]:
# Create the initial data
sales_data = [(1, "2023-10-01", 101, 2, 25.50),
(2, "2023-10-01", 102, 1, 50.00),
(3, "2023-10-02", 101, 1, 12.75)
]
columns = ["Transaction_ID", "Date", "Customer_ID", "Quantity", "Total_Amount"]
# Create a Spark DataFrame
df = spark.createDataFrame(sales_data, columns)
# Write the DataFrame to a Delta table in your Lakehouse
df.write.format("delta").mode("overwrite").save("Tables/dbo/Sales_Transactions")
print("Successfully created the Sales_Transactions table.")

StatementMeta(, 2d0acf99-8737-47bf-915a-f68d70def813, 5, Finished, Available, Finished, False)

Successfully created the Sales_Transactions table.


In [5]:
# New data with an extra column 'Promotion_Code'
new_sales_data = [
(4, "2023-10-03", 103, 3, 75.00, "FALL20"),
(5, "2023-10-03", 102, 1, 45.00, "SAVE10")
]
new_columns = ["Transaction_ID", "Date", "Customer_ID", "Quantity", "Total_Amount", "Promotion_Code"]
# Create a new DataFrame
new_df = spark.createDataFrame(new_sales_data, new_columns)
# Append the new data and allow the schema to merge
new_df.write.format("delta") \
.mode("append") \
.option("mergeSchema", "true") \
.save("Tables/Sales_Transactions")
print("Successfully appended new data with schema evolution.")

StatementMeta(, 2d0acf99-8737-47bf-915a-f68d70def813, 7, Finished, Available, Finished, False)

Successfully appended new data with schema evolution.


In [6]:
%%sql
SELECT * FROM Sales_Transactions

StatementMeta(, 2d0acf99-8737-47bf-915a-f68d70def813, 8, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 5 fields>

In [7]:
# Read the existing table
df = spark.read.format("delta").load("Tables/Sales_Transactions")
# Update the schema: cast Total_Amount to DECIMAL(10,2)
from pyspark.sql.functions import col
df_updated = df.withColumn("Total_Amount", col("Total_Amount").cast("decimal(10,2)"))
# Overwrite the table with the updated schema
df_updated.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("Tables/Sales_Transactions")
print("Schema updated: Total_Amount column now set to DECIMAL(10,2).")

StatementMeta(, 2d0acf99-8737-47bf-915a-f68d70def813, 9, Finished, Available, Finished, False)

Schema updated: Total_Amount column now set to DECIMAL(10,2).


In [8]:
# Check schema
df_updated.printSchema()

StatementMeta(, 2d0acf99-8737-47bf-915a-f68d70def813, 10, Finished, Available, Finished, False)

root
 |-- Transaction_ID: long (nullable = true)
 |-- Date: string (nullable = true)
 |-- Customer_ID: long (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Total_Amount: decimal(10,2) (nullable = true)
 |-- Promotion_Code: string (nullable = true)



In [10]:
%%sql
DESCRIBE Sales_Transactions;

StatementMeta(, 2d0acf99-8737-47bf-915a-f68d70def813, 12, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 3 fields>